# Retrieval Evaluation Playbook: FTS vs Vector vs Hybrid (LanceDB)

Этот ноутбук предназначен для оценки качества поиска в RAG-сервисе. Мы сравним три подхода:
- **Full Text Search (FTS / BM25)**
- **Vector Search (FastEmbed / BAAI/bge-small-en-v1.5)**
- **Hybrid Search (FTS + Vector)**

In [3]:
import json
import os
import re
import time
from dotenv import load_dotenv
from fastembed import TextEmbedding
import lancedb
from lancedb.index import FTS
import matplotlib.pyplot as plt
from openai import OpenAI
import pandas as pd
from tqdm.auto import tqdm

load_dotenv()

True

## 1. Подключение к LanceDB S3/MinIO

In [4]:
TABLE_NAME = os.getenv("TABLE_NAME", "pdf_vectors")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "BAAI/bge-small-en-v1.5")
S3_BUCKET_NAME = os.getenv("S3_BUCKET_NAME", "")
S3_ACCESS_KEY = os.getenv("S3_ACCESS_KEY", "")
S3_SECRET_KEY = os.getenv("S3_SECRET_KEY", "")
S3_ENDPOINT_URL = os.getenv("S3_ENDPOINT_URL", "https://storage.yandexcloud.net")
AWS_REGION = os.getenv("AWS_REGION", "ru-central1")

s3_uri = f"s3://{S3_BUCKET_NAME}/lancedb"
storage_options = {
    "aws_access_key_id": S3_ACCESS_KEY,
    "aws_secret_access_key": S3_SECRET_KEY,
    "aws_region": AWS_REGION,
    "aws_endpoint": S3_ENDPOINT_URL,
    "allow_http": "true" if S3_ENDPOINT_URL.startswith("http://") else "false",
}

db = lancedb.connect(s3_uri, storage_options=storage_options)
table = db.open_table(TABLE_NAME)

print(f"Connected to table '{TABLE_NAME}'. Total rows: {table.count_rows()}")
embedder = TextEmbedding(model_name=EMBEDDING_MODEL)

Connected to table 'pdf_vectors'. Total rows: 1117


## 2. Генерация Ground Truth датасета (вопросы к чанкам)

Берем репрезентативную выборку чанков из LanceDB и генерируем синтетические вопросы через LLM (OpenAI API / VLLM / Ollama).

In [5]:
# Извлекаем документы для оценки
raw_docs = table.search().limit(200).to_list()
documents = [{"text": d["text"], "metadata": d["metadata"]} for d in raw_docs]

client = OpenAI() # Использует OPENAI_API_KEY из .env

prompt_template = """
You are an expert building an evaluation dataset for a RAG system.
Formulate 3 distinct and specific questions that can be directly answered by the text chunk below.
Do not invent facts not present in the chunk. 

Text chunk:
{text}

Provide the output in valid JSON object with key 'questions':
{{"questions": ["question1", "question2", "question3"]}}
""".strip()

def generate_questions(text):
    prompt = prompt_template.format(text=text)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

ground_truth_records = []
print("Generating questions for evaluation...")

for doc in tqdm(documents[:50]):  # Генерируем для первых 50 чанков (150 вопросов)
    try:
        res = generate_questions(doc["text"])
        for q in res.get("questions", []):
            ground_truth_records.append({
                "question": q,
                "target_metadata": doc["metadata"],
                "target_text": doc["text"]
            })
    except Exception as e:
        print(f"Error generating for metadata {doc['metadata']}: {e}")

df_gt = pd.DataFrame(ground_truth_records)
df_gt.to_csv("../data/ground_truth_eval.csv", index=False)
print(f"Created {len(df_gt)} ground truth evaluation samples.")
df_gt.head()

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

## 3. Определение методов поиска

In [ ]:
def search_vector(query: str, k: int = 5):
    query_vector = next(embedder.embed([query])).tolist()
    results = table.search(query_vector, query_type="vector").select(["text", "metadata"]).limit(k).to_list()
    return results

def search_fts(query: str, k: int = 5):
    results = table.search(query, query_type="fts").select(["text", "metadata"]).limit(k).to_list()
    return results

def search_hybrid(query: str, k: int = 5):
    query_vector = next(embedder.embed([query])).tolist()
    try:
        results = (
            table.search(query_vector, query_type="hybrid")
            .text(query)
            .select(["text", "metadata"])
            .limit(k)
            .to_list()
        )
    except Exception:
        # Фолбэк на вектрный, если FTS индекс не доступен в момент вызова
        results = table.search(query_vector).select(["text", "metadata"]).limit(k).to_list()
    return results

## 4. Метрики качества (Hit Rate & MRR)

In [ ]:
def evaluate_search(search_fn, gt_df: pd.DataFrame, k: int = 5):
    hit_count = 0
    mrr_sum = 0.0
    latencies = []

    for _, row in tqdm(gt_df.iterrows(), total=len(gt_df)):
        question = row["question"]
        target_meta = row["target_metadata"]
        
        start_time = time.perf_counter()
        results = search_fn(question, k=k)
        latencies.append(time.perf_counter() - start_time)
        
        # Проверяем позицию правильного документа по метаданным
        retrieved_metas = [r.get("metadata") for r in results]
        
        if target_meta in retrieved_metas:
            hit_count += 1
            rank = retrieved_metas.index(target_meta) + 1
            mrr_sum += 1.0 / rank

    hit_rate = hit_count / len(gt_df)
    mrr = mrr_sum / len(gt_df)
    avg_latency = (sum(latencies) / len(latencies)) * 1000  # ms
    
    return {
        "hit_rate": round(hit_rate, 4),
        "mrr": round(mrr, 4),
        "avg_latency_ms": round(avg_latency, 2)
    }

## 5. Запуск бенчмарка и сравнение результатов

In [ ]:
TOP_K = 5

print("Evaluating Vector Search...")
metrics_vector = evaluate_search(search_vector, df_gt, k=TOP_K)

print("Evaluating Full-Text Search (FTS)...")
metrics_fts = evaluate_search(search_fts, df_gt, k=TOP_K)

print("Evaluating Hybrid Search...")
metrics_hybrid = evaluate_search(search_hybrid, df_gt, k=TOP_K)

# Сводная таблица результатов
df_metrics = pd.DataFrame([
    {"Method": "Vector Search", **metrics_vector},
    {"Method": "Full-Text Search (FTS)", **metrics_fts},
    {"Method": "Hybrid Search", **metrics_hybrid}
])

df_metrics

## 6. Визуализация результатов

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# График Hit Rate и MRR
df_metrics.plot(x="Method", y=["hit_rate", "mrr"], kind="bar", ax=ax[0], rot=0)
ax[0].set_title(f"Retrieval Quality Comparison (Top-{TOP_K})")
ax[0].set_ylabel("Score")
ax[0].set_ylim(0, 1.0)
ax[0].grid(axis="y", linestyle="--", alpha=0.7)

# График Latency
df_metrics.plot(x="Method", y="avg_latency_ms", kind="bar", color="orange", ax=ax[1], rot=0)
ax[1].set_title("Average Search Latency")
ax[1].set_ylabel("Latency (ms)")
ax[1].grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()